# Step 1 — EDA

Plan `docs/PLAN.md` Step 1, acceptance criteria AC-1..AC-3.

Dataset: ULB / Kaggle `mlg-ulb/creditcardfraud`. `data/creditcard.csv` is
gitignored — see the README for how to fetch it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import hour_of_day, stratified_split_60_20_20

sns.set_theme(style="whitegrid")
C_REVIEW = 3.0

# Anchor to the project root via the importable module, so this works
# whether the kernel cwd is /work or /work/notebooks.
from pathlib import Path
import preprocessing
ROOT = Path(preprocessing.__file__).resolve().parent
DATA = ROOT / "data" / "creditcard.csv"

df = pd.read_csv(DATA)
print(DATA)
df.shape

## AC-1 — integrity asserts

Fail loudly if the dataset is not the one the plan was written against.

In [ ]:
assert df.shape[0] == 284_807, df.shape
assert int(df.Class.sum()) == 492, df.Class.sum()
assert df.isnull().sum().sum() == 0

print(f"rows        : {df.shape[0]:,}")
print(f"columns     : {df.shape[1]}")
print(f"frauds      : {int(df.Class.sum())} ({df.Class.mean()*100:.3f}%)")
print(f"nulls       : {int(df.isnull().sum().sum())}")
print(f"Time span   : {df.Time.max():,.0f} s = {df.Time.max()/3600:.1f} h")

## Class distribution

The imbalance is the reason accuracy is useless here.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
counts = df.Class.value_counts().sort_index()

sns.barplot(x=["legit (0)", "fraud (1)"], y=counts.values, ax=ax[0], palette=["#4C78A8", "#E45756"])
ax[0].set_title("Class counts (linear) — fraud is invisible")
ax[0].set_ylabel("transactions")

sns.barplot(x=["legit (0)", "fraud (1)"], y=counts.values, ax=ax[1], palette=["#4C78A8", "#E45756"])
ax[1].set_yscale("log")
ax[1].set_title("Class counts (log) — 578:1")
ax[1].set_ylabel("transactions (log)")
plt.tight_layout(); plt.show()

print(f"imbalance ratio: {counts[0]/counts[1]:.0f} : 1")
print(f"a model predicting 'never fraud' scores {(1-df.Class.mean())*100:.2f}% accuracy")

## AC-3 — Amount distribution by class

This is where the earlier assumption was wrong. The plan predicted fraud amounts
were smaller *on average*; the mean says otherwise, the median agrees, and the
truth is that the distribution is bimodal in cost terms.

In [ ]:
f = df.loc[df.Class == 1, "Amount"]
l = df.loc[df.Class == 0, "Amount"]

summary = pd.DataFrame({
    "fraud": [len(f), f.mean(), f.median(), f.std(), f.max()],
    "legit": [len(l), l.mean(), l.median(), l.std(), l.max()],
}, index=["count", "mean", "median", "std", "max"])
display(summary.round(2))

print(f"mean  : fraud EUR{f.mean():.2f} vs legit EUR{l.mean():.2f}  -> fraud LARGER")
print(f"median: fraud EUR{f.median():.2f} vs legit EUR{l.median():.2f}  -> fraud SMALLER")
print("the tail, not the typical transaction, drives the mean")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))

bins = np.logspace(0, np.log10(max(f.max(), l.max())), 50)
ax[0].hist(l[l > 0], bins=bins, alpha=.6, label="legit", color="#4C78A8", density=True)
ax[0].hist(f[f > 0], bins=bins, alpha=.6, label="fraud", color="#E45756", density=True)
ax[0].set_xscale("log"); ax[0].legend()
ax[0].set_title("Amount by class (log scale, density)")
ax[0].set_xlabel("Amount (EUR)")

ax[1].axvline(C_REVIEW, color="black", ls="--", lw=1.2, label=f"review cost EUR{C_REVIEW:.0f}")
ax[1].hist(f, bins=np.linspace(0, 100, 50), color="#E45756", alpha=.8)
ax[1].legend()
ax[1].set_title("Fraud amounts under EUR100 — most sit left of the review cost")
ax[1].set_xlabel("Amount (EUR)")
plt.tight_layout(); plt.show()

## 🎯 The headline finding

Under a cost objective, a fraud is only worth catching if it is worth more than
the review that catches it.

In [ ]:
below = int((f < C_REVIEW).sum())
zero = int((f == 0).sum())

print(f"frauds worth less than the EUR{C_REVIEW:.0f} review fee : {below} of {len(f)}  ({below/len(f)*100:.0f}%)")
print(f"frauds of exactly EUR0                        : {zero}")
print(f"total value of those {below} frauds            : EUR{f[f < C_REVIEW].sum():,.2f}")
print(f"cost to review them anyway                    : EUR{below*C_REVIEW:,.2f}")
print()
print(f"=> reviewing them destroys EUR{below*C_REVIEW - f[f < C_REVIEW].sum():,.2f} of value")
print("=> a cost-optimal policy ignores ~42% of all fraud ON PURPOSE")
print("=> Policy E does this automatically; a global threshold cannot express it")

## Fraud rate by hour of day

Uses `hour_of_day` from `preprocessing.py` (tested to wrap correctly at the day boundary).

In [ ]:
df["hour"] = hour_of_day(df.Time.values)
by_hour = df.groupby("hour").Class.agg(["mean", "sum", "count"])

fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(by_hour.index, by_hour["mean"] * 100, color="#E45756")
ax.axhline(df.Class.mean() * 100, color="black", ls="--", lw=1, label="overall rate")
ax.set_xlabel("hour of day"); ax.set_ylabel("fraud rate (%)"); ax.legend()
ax.set_title("Fraud rate by hour — night hours run well above baseline")
plt.tight_layout(); plt.show()

peak = by_hour["mean"].idxmax()
print(f"peak hour {peak:02d}:00 at {by_hour.loc[peak,'mean']*100:.2f}% "
      f"({by_hour.loc[peak,'mean']/df.Class.mean():.1f}x the overall rate)")

## Verified constants and baselines

These feed the cost model — record them once, do not re-derive.

In [ ]:
total_fraud = f.sum()
flag_all = len(df) * C_REVIEW
cv = f.std() / f.mean()

print(f"total fraud Amount (flag nothing) : EUR{total_fraud:>12,.2f}")
print(f"flag everything                   : EUR{flag_all:>12,.2f}   ({flag_all/total_fraud:.1f}x worse)")
print()
print(f"fraud Amount CV                   : {cv:.3f}")
print(f"1 + CV^2                          : {1+cv**2:.2f}")
print(f"n_eff for 98 test frauds          : {98/(1+cv**2):.1f}")
print()
print("n_eff ~18 is why the plan reports a paired bootstrap CI rather than")
print("declaring a winner: one large fraud moving in or out of the test set")
print("shifts total cost by more than the gap between models.")

## AC-2 — imbalance in *each* split

The rubric asks this explicitly. Stratification is what makes the answer stable.

In [ ]:
y = df.Class.values
train, val, test = stratified_split_60_20_20(y, random_state=42)

rows = []
for name, idx in [("train", train), ("val", val), ("test", test)]:
    rows.append({
        "split": name,
        "rows": len(idx),
        "frauds": int(y[idx].sum()),
        "fraud rate %": round(y[idx].mean() * 100, 4),
        "fraud Amount EUR": round(df.Amount.values[idx][y[idx] == 1].sum(), 2),
    })
rows.append({"split": "ALL", "rows": len(y), "frauds": int(y.sum()),
             "fraud rate %": round(y.mean()*100, 4), "fraud Amount EUR": round(total_fraud, 2)})
display(pd.DataFrame(rows))
print("fraud rate is preserved across all three splits -> stratification working")

## Temporal split feasibility (T3)

The plan's R8 worried day 2 would hold too few frauds. It does not.

In [ ]:
d1 = df[df.Time < 86_400]
d2 = df[df.Time >= 86_400]

for name, d in [("day 1", d1), ("day 2", d2)]:
    print(f"{name}: {len(d):>7,} rows, {int(d.Class.sum()):>3} frauds, "
          f"EUR{d.loc[d.Class==1,'Amount'].sum():>9,.2f}")
print()
print(f"day 2 has {int(d2.Class.sum())} frauds vs ~{int(len(test)*df.Class.mean())} in a random 20% test split")
print(f"-> {int(d2.Class.sum())/(len(test)*df.Class.mean()):.1f}x more. R8 closed: the temporal check is viable.")

## Caveat for the report

`V1`–`V28` are PCA components fitted on **all 48 hours**. A temporal split
therefore cannot eliminate look-ahead — the future is already baked into the
feature basis before we touch the data. Report the day1→day2 run as a
**distribution-shift check**, never as a leakage remedy.